In [1]:
# Clean, isolated installation to bypass Colab version conflicts
print("Installing core document processing tools...")
!pip install --quiet langchain langchain-community pypdf

print("Installing vector database components...")
!pip install --quiet faiss-cpu sentence-transformers

print("Installing Hugging Face hardware acceleration tools...")
!pip install --quiet langchain-huggingface accelerate

print("\n🎉 Installation check complete! Everything is ready.")

Installing core document processing tools...
Installing vector database components...
Installing Hugging Face hardware acceleration tools...

🎉 Installation check complete! Everything is ready.


In [4]:
import os
import pypdf
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

print("--- STEP 1: DOWNLOADING RESEARCH PAPER ---")
# 1. Have Colab download the PDF directly from the internet (Lightning fast!)
!wget -q -O ai_research_paper.pdf https://arxiv.org/pdf/1706.03762.pdf
print("-> Download complete!\n")

file_name = "ai_research_paper.pdf"
print(f"Processing '{file_name}'...")

# 2. Extract text page-by-page using native pypdf
reader = pypdf.PdfReader(file_name)
raw_pages = []

for page_num, page in enumerate(reader.pages):
    text = page.extract_text()
    if text:
        doc = Document(page_content=text, metadata={"page": page_num + 1, "source": file_name})
        raw_pages.append(doc)

print(f"-> Successfully extracted {len(raw_pages)} pages.")

# 3. Chop the text into AI-readable chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len
)

chunks = text_splitter.split_documents(raw_pages)
print(f"-> Sliced the document into {len(chunks)} individual chunks.")


--- STEP 1: DOWNLOADING RESEARCH PAPER ---
-> Download complete!

Processing 'ai_research_paper.pdf'...
-> Successfully extracted 15 pages.
-> Sliced the document into 52 individual chunks.


In [7]:
# 0. Bulletproof Install: Force Colab to load ALL the database and embedding tools
!pip install -q langchain-huggingface langchain-community faiss-cpu sentence-transformers

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

print("--- STEP 2: BUILDING THE VECTOR DATABASE ---")
print("Converting text chunks into mathematical embeddings (This takes 10-20 seconds)...")

# 1. Download a fast, free embedding model from Hugging Face
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# 2. Create the FAISS Vector Database and insert our chunks
vector_database = FAISS.from_documents(chunks, embeddings)

print("-> Vector Database built successfully! The AI can now instantly search the paper.")

# Let's do a quick test search to prove it works!
test_query = "What hardware was used to train the model?"
print(f"\n--- TEST SEARCH: '{test_query}' ---")

# Search the database for the 2 most mathematically relevant chunks
found_chunks = vector_database.similarity_search(test_query, k=2)

for i, chunk in enumerate(found_chunks):
    print(f"\n[Result {i+1}]")
    print(chunk.page_content)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 37.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 57.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


/tmp/ipykernel_1357/2457245865.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


--- STEP 2: BUILDING THE VECTOR DATABASE ---
Converting text chunks into mathematical embeddings (This takes 10-20 seconds)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

-> Vector Database built successfully! The AI can now instantly search the paper.

--- TEST SEARCH: 'What hardware was used to train the model?' ---

[Result 1]
Table 2: The Transformer achieves better BLEU scores than previous state-of-the-art models on the
English-to-German and English-to-French newstest2014 tests at a fraction of the training cost.
Model
BLEU Training Cost (FLOPs)
EN-DE EN-FR EN-DE EN-FR
ByteNet [18] 23.75
Deep-Att + PosUnk [39] 39.2 1.0 · 1020
GNMT + RL [38] 24.6 39.92 2.3 · 1019 1.4 · 1020
ConvS2S [9] 25.16 40.46 9.6 · 1018 1.5 · 1020
MoE [32] 26.03 40.56 2.0 · 1019 1.2 · 1020
Deep-Att + PosUnk Ensemble [39] 40.4 8.0 · 1020
GNMT + RL Ensemble [38] 26.30 41.16 1.8 · 1020 1.1 · 1021
ConvS2S Ensemble [9] 26.36 41.29 7.7 · 1019 1.2 · 1021
Transformer (base model) 27.3 38.1 3.3 · 1018
Transformer (big) 28.4 41.8 2.3 · 1019
Residual Dropout We apply dropout [33] to the output of each sub-layer, before it is added to the
sub-layer input and normalized. In addition, we ap

In [10]:
# CELL 4: Just install the missing library
!pip install -q langchain transformers accelerate langchain-huggingface
print("✅ Libraries installed successfully!")

✅ Libraries installed successfully!


In [12]:
from transformers import pipeline

print("Loading AI (this takes about 1 minute)...")

# 1. Load TinyLlama directly (Bypassing LangChain completely!)
pipe = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    max_new_tokens=200,
    device_map="auto"
)

# 2. Define our question
question = "Based on the text, what is the 'Transformer' model?"
print(f"\nAsking AI: '{question}'...\n")

# 3. MANUAL RAG STEP A: Search the database for the 2 best paragraphs
docs = vector_database.similarity_search(question, k=2)
context = "\n\n".join([doc.page_content for doc in docs])

# 4. MANUAL RAG STEP B: Build the prompt exactly how TinyLlama likes it
prompt = f"""<|system|>
You are a helpful AI research assistant. Read the following context from an academic paper and answer the question.
Context:
{context}</s>
<|user|>
{question}</s>
<|assistant|>
"""

# 5. Generate the answer!
answer = pipe(prompt)

print("--- AI RESPONSE ---")
print(answer[0]['generated_text'].split("<|assistant|>\n")[-1].strip())

Loading AI (this takes about 1 minute)...


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Asking AI: 'Based on the text, what is the 'Transformer' model?'...



[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer LlamaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


--- AI RESPONSE ---
The 'Transformer' model is a sequence-to-sequence neural network architecture developed by Vaswani et al. (2017). It consists of a stack of encoder and decoder layers, with each layer consisting of a multi-head self-attention mechanism and a simple, position-wise feed-forward network. The Transformer is particularly suited to sequence modeling as it can capture long-range dependencies, which are crucial for tasks such as natural language processing (NLP) and machine translation (MT).


In [13]:
# 0. Install the UI library
!pip install -q gradio

import gradio as gr

print("--- STEP 4: LAUNCHING THE WEB UI ---")

# 1. We wrap your working pipeline into a single function
def chat_with_assistant(message, history):
    # A. Search the database
    docs = vector_database.similarity_search(message, k=2)
    context = "\n\n".join([doc.page_content for doc in docs])

    # B. Build the prompt
    prompt = f"""<|system|>
You are a helpful AI research assistant. Read the following context from an academic paper and answer the question.
Context:
{context}</s>
<|user|>
{message}</s>
<|assistant|>
"""

    # C. Generate and clean the answer
    response = pipe(prompt)
    final_answer = response[0]['generated_text'].split("<|assistant|>\n")[-1].strip()

    return final_answer

# 2. Build the User Interface
demo = gr.ChatInterface(
    fn=chat_with_assistant,
    title="🤖 AI Research Assistant",
    description="Ask me anything about the 'Attention Is All You Need' paper!",
    theme="soft",
    examples=[
        "What is the Transformer model?",
        "What hardware did they use to train it?",
        "Why is self-attention better than recurrent layers?"
    ]
)

# 3. Launch it! (share=True creates a public web link you can send to friends)
demo.launch(share=True)

--- STEP 4: LAUNCHING THE WEB UI ---


/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a44c6723f1d4880b34.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [15]:
import gradio as gr

print("--- STEP 4: LAUNCHING THE UPGRADED WEB UI ---")

# 1. Cleaned up RAG function: Uses Gradio's built-in chat history directly
def premium_chat(message, history):
    # A. Search the vector database
    docs = vector_database.similarity_search(message, k=2)
    context = "\n\n".join([doc.page_content for doc in docs])

    # B. Format the source text for the left sidebar
    source_preview = ""
    for i, doc in enumerate(docs):
        page_num = doc.metadata.get('page', 'Unknown')
        source_preview += f"--- CHUNK {i+1} (Page {page_num}) ---\n{doc.page_content[:400]}...\n\n"

    # C. Build the prompt for TinyLlama
    prompt = f"""<|system|>
You are a helpful AI research assistant. Read the following context from an academic paper and answer the question.
Context:
{context}</s>
<|user|>
{message}</s>
<|assistant|>
"""

    # D. Generate the answer
    response = pipe(prompt)
    final_answer = response[0]['generated_text'].split("<|assistant|>\n")[-1].strip()

    # E. Append to history and return both the conversation and the sources
    history.append((message, final_answer))
    return history, source_preview

# 2. Construct the layout using stable Gradio Blocks elements
with gr.Blocks(theme=gr.themes.Soft()) as demo:

    # Page Header Banner
    gr.HTML("""
        <div style='text-align: center; margin-bottom: 20px;'>
            <h1 style='font-size: 2.2rem; margin-bottom: 5px;'>🤖 Next-Gen AI Research Workspace</h1>
            <p style='font-size: 1.1rem; opacity: 0.8;'>RAG Engine powered by FAISS & TinyLlama-1.1B</p>
        </div>
    """)

    # Main Dashboard Workspace Layout
    with gr.Row():

        # Left Side: Meta Insights & Source Inspector
        with gr.Column(scale=1):
            gr.Markdown("### 📊 Document Metadata")
            gr.HTML(f"""
                <div style='padding: 12px; border: 1px solid #ddd; border-radius: 8px; line-height: 1.6;'>
                    <b>Active File:</b> ai_research_paper.pdf<br>
                    <b>Database Index:</b> FAISS Vector Space<br>
                    <b>Total Chunks:</b> {len(chunks)} Chunks<br>
                    <b>Target GPU:</b> NVIDIA T4 (Active)
                </div>
            """)

            gr.Markdown("### 🔍 RAG Source Context")
            source_box = gr.Textbox(
                label="Retrieved Paragraphs Used by AI",
                placeholder="The exact segments your database discovers will appear here after you ask a question...",
                lines=12,
                interactive=False
            )

        # Right Side: The Conversation Window
        with gr.Column(scale=2):
            chatbot = gr.Chatbot(label="Conversational Engine", height=450)

            with gr.Row():
                msg_input = gr.Textbox(
                    label="Ask a question about the research paper:",
                    placeholder="Type your question here (e.g., What is self-attention?) and press Enter...",
                    scale=4
                )
                submit_btn = gr.Button("Send", variant="primary", scale=1)

            clear_btn = gr.Button("🔄 Clear Conversation Thread", variant="secondary")

    # 3. Handle interactions natively using the chatbot component as its own history state
    submit_event = msg_input.submit(
        fn=premium_chat,
        inputs=[msg_input, chatbot],
        outputs=[chatbot, source_box]
    ).then(lambda: "", outputs=msg_input)

    submit_btn.click(
        fn=premium_chat,
        inputs=[msg_input, chatbot],
        outputs=[chatbot, source_box]
    ).then(lambda: "", outputs=msg_input)

    # Clear button resets the chatbot interface and source box
    clear_btn.click(lambda: ([], ""), outputs=[chatbot, source_box])

# Launch the workspace with a shareable public tunnel link
demo.launch(share=True)

--- STEP 4: LAUNCHING THE UPGRADED WEB UI ---


/tmp/ipykernel_1357/3105337448.py:36: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:
/tmp/ipykernel_1357/3105337448.py:71: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(label="Conversational Engine", height=450)
/tmp/ipykernel_1357/3105337448.py:71: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(label="Conversational Engine", height=450)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e3a0966a0a0565c9c9.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
